# 04. OTP specification curve and sensitivity analysis

Crosses six methodological dimensions to produce 1,080 defensible OTP estimates per route, then runs sensitivity decomposition and generates the specification curve plot.

The framework follows specification-curve analysis (Simonsohn et al. 2020) and multiverse analysis (Steegen et al. 2016), adapted to descriptive KPI measurement. There's no joint inferential test, since OTP is a descriptive metric rather than a hypothesis test. Just the descriptive components: distribution of values across reasonable specifications, plus a per-dimension sensitivity decomposition.

**Prerequisites:** `trip_delays_closest`, `trip_delays_last_before`, `trip_delays_latest` from notebook 03.

**Outputs:** `otp_specifications` table, `docs/specification_curve.png`.

In [ ]:
import duckdb
import pandas as pd
import itertools
from pathlib import Path

PROJECT_ROOT = Path(r'C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework')
DB_PATH      = PROJECT_ROOT / 'data' / 'transit_kpi.duckdb'

con = duckdb.connect(str(DB_PATH))

print(f'Database: {DB_PATH}')
print()
print('Tables:')
print(con.sql('SHOW TABLES').df().to_string(index=False))

In [ ]:
required = ['trip_delays_closest', 'trip_delays_last_before', 'trip_delays_latest']
existing = set(con.sql('SHOW TABLES').df()['name'].tolist())
missing = [t for t in required if t not in existing]
if missing:
    raise RuntimeError(
        f'Missing required tables: {missing}. Run notebook 03 first.'
    )

print('Snapshot rule tables:')
for t in required:
    n = con.sql(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
    print(f'  {t:30s} {n:>10,} rows')

print()
print('Date coverage check (hours of archived data per service_date):')
print(con.sql("""
    SELECT
        service_date,
        ROUND((MAX(predicted_unix) - MIN(predicted_unix)) / 3600.0, 1) AS span_hours
    FROM trip_delays_latest
    GROUP BY service_date
    ORDER BY service_date
""").df().to_string(index=False))

## Define the specification dimensions

Six dimensions, fully crossed:

| Dimension | Variants |
|-----------|----------|
| `snapshot_rule` | `closest`, `last_before`, `latest` |
| `lateness_window` | `0_3`, `neg1_5`, `neg2_7` |
| `early_treatment` | `count_early_as_late`, `ignore_early` |
| `stop_selection` | `all_stops`, `timepoints_only`, `terminals_only` |
| `time_of_day` | `am_peak`, `midday`, `pm_peak`, `evening`, `all_day` |
| `delay_basis` | `arrival`, `departure` |

3 × 3 × 2 × 3 × 5 × 2 × 2 routes = 1,080 specifications.

Every variant is theoretically justified, statistically valid, and non-redundant. Variants that are clearly inferior to alternatives are not included (e.g., a +0/+10 window with no negative tail).

In [ ]:
snapshot_rules = [
    {'name': 'closest',     'table': 'trip_delays_closest'},
    {'name': 'last_before', 'table': 'trip_delays_last_before'},
    {'name': 'latest',      'table': 'trip_delays_latest'},
]

lateness_windows = [
    {'name': '0_3',     'lower':  0, 'upper': 3},
    {'name': 'neg1_5',  'lower': -1, 'upper': 5},
    {'name': 'neg2_7',  'lower': -2, 'upper': 7},
]

early_treatments = [
    {'name': 'count_early_as_late'},
    {'name': 'ignore_early'},
]

stop_selections = [
    {'name': 'all_stops',       'sql': None},
    {'name': 'timepoints_only', 'sql': 'timepoint = 1'},
    {'name': 'terminals_only',  'sql': (
        'stop_sequence IN ('
        '  SELECT MIN(stop_sequence) FROM stop_times st WHERE st.trip_id = td.trip_id'
        ') OR stop_sequence IN ('
        '  SELECT MAX(stop_sequence) FROM stop_times st WHERE st.trip_id = td.trip_id'
        ')'
    )},
]

times_of_day = [
    {'name': 'am_peak', 'sql': 'scheduled_arrival_seconds >= 25200 AND scheduled_arrival_seconds < 32400'},
    {'name': 'midday',  'sql': 'scheduled_arrival_seconds >= 32400 AND scheduled_arrival_seconds < 57600'},
    {'name': 'pm_peak', 'sql': 'scheduled_arrival_seconds >= 57600 AND scheduled_arrival_seconds < 64800'},
    {'name': 'evening', 'sql': 'scheduled_arrival_seconds >= 64800 AND scheduled_arrival_seconds < 75600'},
    {'name': 'all_day', 'sql': None},
]

delay_bases = [
    {'name': 'arrival'},
    {'name': 'departure'},
]

routes = ['23', '47']

specs = []
for sr, lw, et, ss, tod, db, rt in itertools.product(
    snapshot_rules, lateness_windows, early_treatments,
    stop_selections, times_of_day, delay_bases, routes
):
    specs.append({
        'snapshot_rule':    sr['name'],
        'snapshot_table':   sr['table'],
        'lateness_window':  lw['name'],
        'window_lower':     lw['lower'],
        'window_upper':     lw['upper'],
        'early_treatment':  et['name'],
        'stop_selection':   ss['name'],
        'stop_sql':         ss['sql'],
        'time_of_day':      tod['name'],
        'tod_sql':          tod['sql'],
        'delay_basis':      db['name'],
        'route_id':         rt,
    })

print(f'Total specifications: {len(specs)}')
print(f'Per route:            {len(specs) // len(routes)}')

## Run all specifications

One SQL query per specification. `build_query` assembles the right `FROM` table, lateness condition, and filters based on the spec dict. Departure-based delay is recomputed on the fly using the offset between `scheduled_departure_seconds` and `scheduled_arrival_seconds` (since `delay_minutes` was computed against arrival in notebook 03).

Each spec is tagged with a sample-size tier:
- `drop`: N < 1,000 events (excluded from the analysis)
- `flag`: 1,000 ≤ N < 5,000 (lower-sample, flagged for caution)
- `ok`: N ≥ 5,000 (primary tier)

In [ ]:
def build_query(spec):
    table = spec['snapshot_table']
    rt    = spec['route_id']
    lo    = spec['window_lower']
    hi    = spec['window_upper']

    if spec['delay_basis'] == 'arrival':
        delay_expr = 'delay_minutes'
    else:
        # Recompute against scheduled departure
        delay_expr = (
            'ROUND((predicted_unix - '
            '(scheduled_unix + scheduled_departure_seconds - scheduled_arrival_seconds))'
            ' / 60.0, 1)'
        )

    if spec['early_treatment'] == 'count_early_as_late':
        on_time_cond = f'{delay_expr} BETWEEN {lo} AND {hi}'
    else:
        on_time_cond = f'{delay_expr} <= {hi}'

    where = [
        f"route_id = '{rt}'",
        'ISODOW(CAST(service_date AS DATE)) BETWEEN 1 AND 5',  # weekdays only
    ]
    if spec['tod_sql']:
        where.append(spec['tod_sql'])
    if spec['stop_sql']:
        where.append(spec['stop_sql'])

    return f"""
        SELECT
            COUNT(*) AS n_events,
            SUM(CASE WHEN {on_time_cond} THEN 1 ELSE 0 END) AS n_on_time
        FROM {table} td
        WHERE {' AND '.join(where)}
    """

results = []
for i, spec in enumerate(specs):
    if i % 100 == 0:
        print(f'  running spec {i}/{len(specs)}...')
    n_events, n_on_time = con.sql(build_query(spec)).fetchone()
    results.append({
        'snapshot_rule':   spec['snapshot_rule'],
        'lateness_window': spec['lateness_window'],
        'early_treatment': spec['early_treatment'],
        'stop_selection':  spec['stop_selection'],
        'time_of_day':     spec['time_of_day'],
        'delay_basis':     spec['delay_basis'],
        'route_id':        spec['route_id'],
        'n_events':        n_events,
        'n_on_time':       n_on_time,
        'otp_pct':         (round(n_on_time * 100.0 / n_events, 2)
                            if n_events > 0 else None),
    })

df = pd.DataFrame(results)

def reliability(n):
    if n is None or n < 1000:
        return 'drop'
    elif n < 5000:
        return 'flag'
    else:
        return 'ok'

df['tier'] = df['n_events'].apply(reliability)

print()
print(f'Total specifications run: {len(df)}')
print('Tier distribution:')
print(df['tier'].value_counts().to_string())

In [ ]:
# Persist tiers 'ok' and 'flag'; drop tier='drop'
kept = df[df['tier'] != 'drop'].copy()

con.execute('DROP TABLE IF EXISTS otp_specifications')
con.execute('CREATE TABLE otp_specifications AS SELECT * FROM kept')

print('otp_specifications table:')
print(con.sql('SELECT COUNT(*) AS rows FROM otp_specifications').df().to_string(index=False))
print()
print('Tier breakdown of kept specifications:')
print(con.sql("""
    SELECT tier, COUNT(*) AS specs
    FROM otp_specifications
    GROUP BY tier
""").df().to_string(index=False))

## Distribution of OTP across defensible specifications

Specification-curve analysis explicitly avoids singling out one specification as 'the' answer; that's the move it's designed to prevent. The headline result is the *distribution*: min, quartiles, median, max, range. If a single number must be cited (paper abstract, etc.), cite the median with a note that it's the median of N defensible specifications.

The slice below holds five dimensions fixed at one defensible reference and varies only the snapshot rule. Useful for showing the magnitude of the snapshot-rule effect, but it's one slice through the spec space, not the answer.

In [ ]:
print('Distribution of OTP across defensible specifications, by route:')
print(con.sql("""
    SELECT
        route_id,
        COUNT(*) AS n_specs,
        ROUND(MIN(otp_pct), 1)                  AS min_otp,
        ROUND(QUANTILE_CONT(otp_pct, 0.25), 1)  AS q25_otp,
        ROUND(MEDIAN(otp_pct), 1)               AS median_otp,
        ROUND(QUANTILE_CONT(otp_pct, 0.75), 1)  AS q75_otp,
        ROUND(MAX(otp_pct), 1)                  AS max_otp,
        ROUND(MAX(otp_pct) - MIN(otp_pct), 1)   AS range_pp,
        ROUND(STDDEV(otp_pct), 1)               AS stddev_otp
    FROM otp_specifications
    WHERE tier = 'ok'
    GROUP BY route_id
    ORDER BY route_id
""").df().to_string(index=False))

print()
print('---')
print('Snapshot-rule slice (other dimensions fixed at neg1_5 + count_early_as_late')
print('+ all_stops + all_day + arrival). One slice, not the answer.')
print()
print(con.sql("""
    SELECT snapshot_rule, route_id, n_events, otp_pct
    FROM otp_specifications
    WHERE lateness_window = 'neg1_5'
      AND early_treatment = 'count_early_as_late'
      AND stop_selection = 'all_stops'
      AND time_of_day = 'all_day'
      AND delay_basis = 'arrival'
    ORDER BY route_id, snapshot_rule
""").df().to_string(index=False))

## Sensitivity decomposition

For each dimension, hold every other dimension fixed and measure the OTP range across that dimension's variants. Average across all combinations. Larger mean range = more sensitivity.

This is descriptive, not a formal ANOVA-style variance decomposition. It corresponds to the dashboard panel of the spec curve plot. Dimensions whose variants spread across the OTP curve get high mean range; dimensions whose variants overlap get low mean range.

In [ ]:
dims = ['snapshot_rule', 'lateness_window', 'early_treatment',
        'stop_selection', 'time_of_day', 'delay_basis']

kept = con.sql("SELECT * FROM otp_specifications WHERE tier = 'ok'").df()

summary = []
for d in dims:
    other_dims = [x for x in dims if x != d] + ['route_id']
    grouped = kept.groupby(other_dims)['otp_pct']
    ranges = grouped.max() - grouped.min()
    summary.append({
        'dimension':       d,
        'mean_range_pp':   round(ranges.mean(), 2),
        'median_range_pp': round(ranges.median(), 2),
        'max_range_pp':    round(ranges.max(), 2),
        'n_combinations':  len(ranges),
    })

summary_df = pd.DataFrame(summary).sort_values('mean_range_pp', ascending=False)
print('Sensitivity by dimension (mean OTP range when varying that dimension):')
print()
print(summary_df.to_string(index=False))

## The specification curve

Two-panel figure adapted from Simonsohn et al. 2020 Figure 2.

- **Top panel:** OTP across all defensible specs, sorted ascending. Each route plotted separately (each route's 432 specs scaled to the full x-range so the shapes align). Median reference lines show the per-route center of the distribution.
- **Bottom panel:** dashboard. For each spec on the x-axis, ticks indicate which variant of each dimension was used. Color-coded by dimension group. Dimensions with clear OTP gradients (snapshot rule, lateness window) show concentrated tick patterns. Dimensions that don't drive variation (stop selection, time of day) show roughly uniform ticks.

`delay_basis` is omitted: SEPTA's static GTFS encodes arrival_time = departure_time at every stop, so the dimension produces zero variation.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle
import numpy as np

kept = con.sql("SELECT * FROM otp_specifications WHERE tier = 'ok' ORDER BY otp_pct").df()

NAVY = '#1a2438'
GOLD = '#E8B547'
TEAL = '#2D9B8A'
BLUE = '#4A7BC2'
RED  = '#C0392B'
PURPLE = '#7B5CA7'

DIM_COLORS = {
    'snapshot_rule':   GOLD,
    'lateness_window': TEAL,
    'early_treatment': BLUE,
    'stop_selection':  RED,
    'time_of_day':     PURPLE,
}

route_23_otp = kept[kept['route_id'] == '23'].sort_values('otp_pct')['otp_pct'].values
route_47_otp = kept[kept['route_id'] == '47'].sort_values('otp_pct')['otp_pct'].values
median_23 = np.median(route_23_otp)
median_47 = np.median(route_47_otp)

kept_sorted = kept.sort_values('otp_pct').reset_index(drop=True)
n_specs = len(kept_sorted)

fig = plt.figure(figsize=(11, 7.5), dpi=150)
fig.patch.set_facecolor('white')

gs = gridspec.GridSpec(2, 1, height_ratios=[2.0, 3.0], hspace=0.08,
                       left=0.20, right=0.97, top=0.93, bottom=0.10)

# Top panel
ax_top = fig.add_subplot(gs[0])
ax_top.set_facecolor('white')

x_23_scaled = np.linspace(0, n_specs - 1, len(route_23_otp))
x_47_scaled = np.linspace(0, n_specs - 1, len(route_47_otp))

ax_top.plot(x_23_scaled, route_23_otp, '.', color=NAVY, alpha=0.7, markersize=4.5,
            label='Route 23')
ax_top.plot(x_47_scaled, route_47_otp, '.', color=GOLD, alpha=0.85, markersize=4.5,
            label='Route 47')

ax_top.axhline(median_23, color=NAVY, alpha=0.35, linestyle='--', linewidth=0.8)
ax_top.axhline(median_47, color=GOLD, alpha=0.55, linestyle='--', linewidth=0.8)

ax_top.text(n_specs + 12, median_23, f'{median_23:.0f}%', fontsize=9, color=NAVY,
            family='serif', fontweight='bold', va='center')
ax_top.text(n_specs + 12, median_47, f'{median_47:.0f}%', fontsize=9, color=GOLD,
            family='serif', fontweight='bold', va='center')

ax_top.text(0.02, 0.97,
            f'Route 23   median {median_23:.1f}%   range {route_23_otp.min():.0f}-{route_23_otp.max():.0f}%\n'
            f'Route 47   median {median_47:.1f}%   range {route_47_otp.min():.0f}-{route_47_otp.max():.0f}%',
            transform=ax_top.transAxes, fontsize=9, family='serif',
            verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.95,
                      edgecolor=NAVY, linewidth=0.5))

ax_top.legend(loc='lower right', fontsize=9, framealpha=0.92,
              ncol=2, handletextpad=0.3, columnspacing=1.0)
ax_top.set_ylabel('OTP (%)', fontsize=11, family='sans-serif', color=NAVY)
ax_top.set_ylim(15, 105)
ax_top.set_xlim(-8, n_specs + 30)
ax_top.grid(True, alpha=0.25, color='#999999', axis='y')
ax_top.spines['top'].set_visible(False)
ax_top.spines['right'].set_visible(False)
ax_top.spines['left'].set_color(NAVY)
ax_top.spines['bottom'].set_color(NAVY)
ax_top.tick_params(colors=NAVY, labelsize=9)
ax_top.set_xticklabels([])
ax_top.set_title('Distribution of OTP across defensible specifications',
                 fontsize=13, family='serif', color=NAVY, fontweight='bold',
                 pad=10)

# Bottom panel: dashboard
ax_bot = fig.add_subplot(gs[1])
ax_bot.set_facecolor('white')

dimensions_order = [
    ('snapshot_rule', ['closest', 'last_before', 'latest']),
    ('lateness_window', ['0_3', 'neg1_5', 'neg2_7']),
    ('early_treatment', ['count_early_as_late', 'ignore_early']),
    ('stop_selection', ['all_stops', 'timepoints_only', 'terminals_only']),
    ('time_of_day', ['am_peak', 'midday', 'pm_peak', 'evening', 'all_day']),
]

y_positions = {}
y_pos = 0
group_labels = []

for dim, variants in dimensions_order:
    group_start = y_pos
    for v in variants:
        y_positions[(dim, v)] = y_pos
        y_pos += 1
    group_labels.append((group_start, y_pos - 1, dim))
    y_pos += 0.7

total_rows = y_pos
x_specs = np.arange(n_specs)

for dim, variants in dimensions_order:
    color = DIM_COLORS[dim]
    for v in variants:
        y = y_positions[(dim, v)]
        mask = (kept_sorted[dim] == v).values
        x_active = x_specs[mask]
        ax_bot.scatter(x_active, [y] * len(x_active),
                       marker='|', s=24, color=color, alpha=0.7,
                       linewidths=0.9)

# Subtle background bands per dimension group
for start, end, dim in group_labels:
    color = DIM_COLORS[dim]
    rect = Rectangle((-8, start - 0.4), n_specs + 38, end - start + 0.8,
                     facecolor=color, alpha=0.06, zorder=0)
    ax_bot.add_patch(rect)

yticks, yticklabels = [], []
for dim, variants in dimensions_order:
    for v in variants:
        yticks.append(y_positions[(dim, v)])
        yticklabels.append(v.replace('_', ' '))

ax_bot.set_yticks(yticks)
ax_bot.set_yticklabels(yticklabels, fontsize=8, family='sans-serif')

for start, end, dim in group_labels:
    mid = (start + end) / 2
    ax_bot.text(-0.21, mid, dim.replace('_', ' '),
                transform=ax_bot.get_yaxis_transform(),
                fontsize=9, family='sans-serif', fontweight='bold',
                color=DIM_COLORS[dim],
                ha='right', va='center')

ax_bot.set_xlabel('Specification (sorted by OTP)', fontsize=11,
                  family='sans-serif', color=NAVY)
ax_bot.set_xlim(-8, n_specs + 30)
ax_bot.set_ylim(-0.6, total_rows - 0.4)
ax_bot.invert_yaxis()
ax_bot.grid(True, alpha=0.2, color='#999999', axis='x', zorder=0)
ax_bot.spines['top'].set_visible(False)
ax_bot.spines['right'].set_visible(False)
ax_bot.spines['left'].set_color(NAVY)
ax_bot.spines['bottom'].set_color(NAVY)
ax_bot.tick_params(colors=NAVY, labelsize=9, left=False)
ax_bot.tick_params(axis='y', pad=2)

fig.text(0.58, 0.015,
         'Note: delay_basis dimension omitted (both variants produce identical OTP in SEPTA data)',
         fontsize=8, family='sans-serif', color='#666666',
         ha='center', va='bottom', style='italic')

out_path = PROJECT_ROOT / 'docs' / 'specification_curve.png'
out_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(out_path, dpi=150, bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved to {out_path}')

## Slice analysis

For each dimension, hold the other five at one defensible reference point and show how OTP varies across just that dimension. Useful for the paper's discussion of each dimension's behavior, but it's one slice, and different reference points may yield different patterns. The full distribution and the sensitivity decomposition above are the more general summaries.

In [ ]:
reference_point = {
    'snapshot_rule':   'latest',
    'lateness_window': 'neg1_5',
    'early_treatment': 'count_early_as_late',
    'stop_selection':  'all_stops',
    'time_of_day':     'all_day',
    'delay_basis':     'arrival',
}

print('Reference point:')
for k, v in reference_point.items():
    print(f'  {k}: {v}')
print()

for varying in reference_point:
    fixed = {k: v for k, v in reference_point.items() if k != varying}
    where_clauses = [f"{k} = '{v}'" for k, v in fixed.items()]
    where_clauses.append("tier = 'ok'")
    where_sql = ' AND '.join(where_clauses)

    print(f'Varying {varying}:')
    print(con.sql(f"""
        SELECT
            {varying}, route_id, n_events, otp_pct
        FROM otp_specifications
        WHERE {where_sql}
        ORDER BY route_id, {varying}
    """).df().to_string(index=False))
    print()

## Ad-hoc queries

Two diagnostics kept around for paper-writing reference.

In [ ]:
# Stop-event count per day of week. Confirms weekday filter is doing what we expect
# and partial-day Saturdays (April 4 and April 25) didn't make it into the analysis.
con.sql("""
    SELECT
        service_date,
        ISODOW(CAST(service_date AS DATE)) AS dow,
        route_id,
        COUNT(*) AS n
    FROM trip_delays_latest
    GROUP BY service_date, route_id
    ORDER BY service_date, route_id
""").df()

In [ ]:
# What dimension combinations produce the extremes? Both ends should share
# patterns rather than looking like noise (sanity check that the spread is
# methodological, not data quality).
print(con.sql("""
    SELECT
        snapshot_rule, lateness_window, early_treatment,
        stop_selection, time_of_day, delay_basis,
        route_id, otp_pct, n_events
    FROM otp_specifications
    WHERE otp_pct >= 95 OR otp_pct <= 25
    ORDER BY otp_pct
""").df().to_string())